# **Feature Engineering**

## Objectives

- Prepare the cleaned Premier League player dataset for machine learning.
- Select appropriate predictor features based on findings from exploratory data analysis.
- Remove features that would cause target leakage.
- Investigate and handle missing values required for modelling.
- Transform categorical variables into a suitable numerical representation.
- Prepare and save the engineered dataset for model development.

## Inputs

- `data/processed/all_players_cleaned.csv` – Cleaned player-season dataset produced by the Data Cleaning notebook.

## Outputs

- A feature-engineered dataset suitable for machine learning and model development.

## Additional Comments

- The `HighScorer` target represents players who scored 10 or more Premier League goals in a season.
- EDA identified substantial class imbalance, correlated attacking statistics and systematic missing values in several shooting features. These findings will be considered when preparing the features for modelling.

### Imports

In [32]:
import os
import pandas as pd
import numpy as np

### Change Working directory

In [33]:
current_dir = os.getcwd()
current_dir

'C:\\code\\premier-league-predictor\\premier-league-predictor'

In [34]:
os.chdir(r"C:\code\premier-league-predictor\premier-league-predictor")
print("You set a new current directory")

You set a new current directory


In [35]:
current_dir = os.getcwd()
current_dir

'C:\\code\\premier-league-predictor\\premier-league-predictor'

In [36]:
df = pd.read_csv("data/processed/all_players_cleaned.csv")

In [37]:
df.shape

(8196, 54)

In [38]:
df.head()

,Name,Position,Appearances,Clean sheets,Goals conceded,Tackles,Tackle success %,Last man tackles,Blocked shots,Interceptions,...,Big chances missed,Saves,Penalties saved,Punches,High Claims,Catches,Sweeper clearances,Throw outs,Goal Kicks,Season
0,Rolando Aarons,Midfielder,10,NaN,NaN,13.0,77.0,NaN,0.0,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015-16
1,Almen Abdi,Midfielder,32,NaN,NaN,83.0,78.0,NaN,10.0,32.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015-16
2,Abdul Rahman Baba,Defender,15,2.0,13.0,47.0,83.0,0.0,1.0,23.0,...,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015-16
3,Mehdi Abeid,Midfielder,0,NaN,NaN,0.0,0.0,NaN,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015-16
4,Tammy Abraham,Forward,2,NaN,NaN,0.0,NaN,NaN,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2015-16


## Define Target Variable

The machine learning model will predict whether a player is classified as a `HighScorer`, defined as scoring 10 or more Premier League goals in a season.

The target variable is recreated from `Goals` before goal-derived features are removed from the predictor dataset.

In [39]:
df.columns

Index(['Name', 'Position', 'Appearances', 'Clean sheets', 'Goals conceded',
       'Tackles', 'Tackle success %', 'Last man tackles', 'Blocked shots',
       'Interceptions', 'Clearances', 'Headed Clearance',
       'Clearances off line', 'Recoveries', 'Duels won', 'Duels lost',
       'Successful 50/50s', 'Aerial battles won', 'Aerial battles lost',
       'Own goals', 'Errors leading to goal', 'Assists', 'Passes',
       'Passes per match', 'Big chances created', 'Crosses',
       'Cross accuracy %', 'Through balls', 'Accurate long balls',
       'Yellow cards', 'Red cards', 'Fouls', 'Offsides', 'Goals',
       'Headed goals', 'Goals with right foot', 'Goals with left foot',
       'Hit woodwork', 'Goals per match', 'Penalties scored',
       'Freekicks scored', 'Shots', 'Shots on target', 'Shooting accuracy %',
       'Big chances missed', 'Saves', 'Penalties saved', 'Punches',
       'High Claims', 'Catches', 'Sweeper clearances', 'Throw outs',
       'Goal Kicks', 'Season'],
     

In [40]:
df["HighScorer"] = df["Goals"] >= 10

In [41]:
df["HighScorer"].value_counts()

HighScorer
False    7984
True      212
Name: count, dtype: int64

### Remove Target Leakage Features

The `HighScorer` target is derived from the number of goals scored by each player. Therefore, `Goals` cannot be used as a predictor because it would directly reveal information used to determine the target.

Other statistics that directly describe how those goals were scored are also excluded to reduce target leakage. The model should instead learn from player characteristics and performance statistics that do not directly provide the outcome being predicted.

In [42]:
goal_columns = [
    "Goals",
    "Headed goals",
    "Goals with right foot",
    "Goals with left foot",
    "Goals per match",
    "Penalties scored",
    "Freekicks scored"
]

In [43]:
goal_columns

['Goals',
 'Headed goals',
 'Goals with right foot',
 'Goals with left foot',
 'Goals per match',
 'Penalties scored',
 'Freekicks scored']

In [44]:
x = df.drop(columns=goal_columns + ["HighScorer"])
y = df["HighScorer"]

In [45]:
x.shape

(8196, 47)

In [46]:
y.shape

(8196,)

In [47]:
x.columns

Index(['Name', 'Position', 'Appearances', 'Clean sheets', 'Goals conceded',
       'Tackles', 'Tackle success %', 'Last man tackles', 'Blocked shots',
       'Interceptions', 'Clearances', 'Headed Clearance',
       'Clearances off line', 'Recoveries', 'Duels won', 'Duels lost',
       'Successful 50/50s', 'Aerial battles won', 'Aerial battles lost',
       'Own goals', 'Errors leading to goal', 'Assists', 'Passes',
       'Passes per match', 'Big chances created', 'Crosses',
       'Cross accuracy %', 'Through balls', 'Accurate long balls',
       'Yellow cards', 'Red cards', 'Fouls', 'Offsides', 'Hit woodwork',
       'Shots', 'Shots on target', 'Shooting accuracy %', 'Big chances missed',
       'Saves', 'Penalties saved', 'Punches', 'High Claims', 'Catches',
       'Sweeper clearances', 'Throw outs', 'Goal Kicks', 'Season'],
      dtype='object')

### Target and Leakage Observations

The `HighScorer` target contains **212 high-scorer records** and **7,984 non-high-scorer records** across 8,196 player-season observations.

Seven goal-derived features were excluded from the predictor dataset to prevent target leakage. The target variable was also separated from the predictors, resulting in **47 candidate predictor features**.

Further feature selection is required before modelling, as not all remaining features are expected to provide useful information for predicting high scorers.

***

## Feature Selection

The predictor dataset currently contains 47 candidate features. Exploratory data analysis showed that not all available statistics are equally relevant to identifying high scorers.

A smaller set of candidate features will therefore be selected based on their relationship with the target, footballing relevance and findings from the exploratory analysis.

In [48]:
x = x.drop(columns=["Name"])

In [49]:
x.shape

(8196, 46)

`Name` is excluded because it is an identifier rather than a generalisable player performance feature.

`Season` is also excluded. Exploratory analysis showed that the proportion of high scorers remained relatively consistent across seasons, and the season itself is not considered necessary for predicting whether a player's performance statistics indicate a high scorer.

In [50]:
x = x.drop(columns=["Season"])

In [51]:
x.shape

(8196, 45)

### Position-Specific Features

Goalkeeper-specific statistics are excluded because they describe actions that are not relevant to the goal-scoring performance of outfield players. Retaining these features would primarily identify goalkeepers rather than provide useful information about attacking performance.

`Position` is retained because exploratory analysis showed a clear relationship between playing position and the `HighScorer` target.

***

### Remove Goalkeeper-Specific Features

The dataset contains several statistics that are specific to goalkeepers, including saves, penalties saved, punches, claims, catches and distribution statistics.

These features are removed because they describe goalkeeper-specific actions rather than attacking performance. Including them would primarily help the model identify goalkeepers rather than provide meaningful information for predicting whether a player will be classified as a high scorer.

The `Position` feature is retained separately because EDA showed a clear relationship between playing position and the `HighScorer` target.

In [ ]:
goalkeeper_columns = [
    "Saves",
    "Penalties saved",
    "Punches",
    "High Claims",
    "Catches",
    "Sweeper clearances",
    "Throw outs",
    "Goal Kicks"
]

In [53]:
x = x.drop(columns=goalkeeper_columns)

In [55]:
x.shape

(8196, 37)

In [56]:
x.columns

Index(['Position', 'Appearances', 'Clean sheets', 'Goals conceded', 'Tackles',
       'Tackle success %', 'Last man tackles', 'Blocked shots',
       'Interceptions', 'Clearances', 'Headed Clearance',
       'Clearances off line', 'Recoveries', 'Duels won', 'Duels lost',
       'Successful 50/50s', 'Aerial battles won', 'Aerial battles lost',
       'Own goals', 'Errors leading to goal', 'Assists', 'Passes',
       'Passes per match', 'Big chances created', 'Crosses',
       'Cross accuracy %', 'Through balls', 'Accurate long balls',
       'Yellow cards', 'Red cards', 'Fouls', 'Offsides', 'Hit woodwork',
       'Shots', 'Shots on target', 'Shooting accuracy %',
       'Big chances missed'],
      dtype='object')

***

### Select Candidate Features

Based on the findings from exploratory data analysis, a smaller set of candidate features is selected for further preparation and modelling.

The selected features focus primarily on playing position, appearances, attacking performance, creativity and player involvement. Defensive and lower-relevance statistics are excluded to reduce unnecessary complexity.

`Passes per match` is retained instead of total `Passes`. Total passes are influenced by the number of appearances a player makes, while passes per match provides a better indication of typical passing involvement. `Appearances` is retained separately to represent playing time.

These features represent an initial EDA-informed selection and may be refined further when considering missing values, correlation and model performance.

In [57]:
selected_features = [
    "Position",
    "Appearances",
    "Blocked shots",
    "Assists",
    "Passes per match",
    "Big chances created",
    "Crosses",
    "Cross accuracy %",
    "Through balls",
    "Offsides",
    "Hit woodwork",
    "Shots",
    "Shots on target",
    "Shooting accuracy %",
    "Big chances missed",
    "Recoveries",
    "Duels won",
    "Duels lost",
    "Successful 50/50s",
    "Aerial battles won",
    "Aerial battles lost"
]

In [58]:
x = x[selected_features]

In [62]:
x.shape

(8196, 21)

In [63]:
x.head()

,Position,Appearances,Blocked shots,Assists,Passes per match,Big chances created,Crosses,Cross accuracy %,Through balls,Offsides,...,Shots,Shots on target,Shooting accuracy %,Big chances missed,Recoveries,Duels won,Duels lost,Successful 50/50s,Aerial battles won,Aerial battles lost
0,Midfielder,10,0.0,1,11.90,1.0,12.0,25.0,0.0,0.0,...,2.0,1.0,50.0,0.0,23.0,29.0,34.0,5.0,4.0,6.0
1,Midfielder,32,10.0,0,29.31,4.0,55.0,31.0,2.0,1.0,...,39.0,10.0,26.0,1.0,137.0,140.0,153.0,12.0,22.0,31.0
2,Defender,15,1.0,1,35.07,0.0,31.0,16.0,0.0,2.0,...,NaN,NaN,NaN,NaN,76.0,67.0,65.0,8.0,8.0,13.0
3,Midfielder,0,0.0,0,0.00,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Forward,2,1.0,0,5.00,0.0,2.0,NaN,NaN,3.0,...,2.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


## Missing Value Assessment

The selected feature set is assessed for missing values before any transformations are applied. Missing values were identified during EDA, but they are reassessed here because only the features selected for modelling now need to be considered.

The percentage of missing values in each selected feature is calculated to help determine an appropriate treatment strategy.

In [65]:
x.isna().mean() * 100

Position                0.000000
Appearances             0.000000
Blocked shots          11.725232
Assists                 0.000000
Passes per match        0.000000
Big chances created     0.000000
Crosses                11.725232
Cross accuracy %       32.613470
Through balls          32.613470
Offsides               11.725232
Hit woodwork            0.000000
Shots                  43.643241
Shots on target        43.643241
Shooting accuracy %    43.643241
Big chances missed     43.643241
Recoveries             32.613470
Duels won              32.613470
Duels lost             32.613470
Successful 50/50s      32.613470
Aerial battles won     32.613470
Aerial battles lost    32.613470
dtype: float64

In [88]:
((df["HighScorer"]) & (df["Shots"].isna())).sum()

np.int64(0)

In [90]:
shooting_columns = [
    "Shots",
    "Shots on target",
    "Shooting accuracy %",
    "Big chances missed"
]

In [93]:
for column in shooting_columns:
    print(column, ((df["HighScorer"]) & (df[column].isna())).sum())


Shots 0
Shots on target 0
Shooting accuracy % 0
Big chances missed 0


### Missing Value Considerations

EDA identified systematic missingness in several selected features, particularly the shooting statistics. These features are retained because they showed strong relationships with the `HighScorer` target despite their missing values.

A further check confirmed that none of the 212 high scorers have missing values for `Shots`, `Shots on target`, `Shooting accuracy %` or `Big chances missed`.

The remaining missing values will therefore be handled during feature preparation rather than removing these potentially informative features.

In [ ]:
for column in shooting_columns:
    print(column, ((df["HighScorer"]) & (df[column].isna())).sum())


Shots 0
Shots on target 0
Shooting accuracy % 0
Big chances missed 0


### Missing Value Considerations

EDA identified systematic missingness in several selected features, particularly the shooting statistics. These features are retained because they showed strong relationships with the `HighScorer` target despite their missing values.

A further check confirmed that none of the 212 high scorers have missing values for `Shots`, `Shots on target`, `Shooting accuracy %` or `Big chances missed`.

The remaining missing values will therefore be handled during feature preparation rather than removing these potentially informative features.